In [0]:
#libraries
from pyspark import pipelines as dp
from pyspark.sql import functions as F

#dlp for bronze
@dp.table(
    name="bronze_turbine_raw",
    comment="Daily wind Farm Data Exactly As Is (how it was transferred by the generator)",
    table_properties={"quality": "bronze"},
)
def bronze_turbine_raw():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "false") # like "N/A" in a numeric column must land in the table, not fail the load. It can be diverted in silver by constraints
        .option("header", "true")
        .load('/Volumes/turbine_poc/wind_farm/landing/raw')
        .select(
            "*",
            F.col("_metadata.file_path").alias("source_file"),
            F.current_timestamp().alias("ingest_ts"),
        )
    )